# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [ ]:
!pip install pinecone "datasets<4.0" librosa panns-inference python-dotenv tqdm groq

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import gc
import io
import os
import random
import time
import urllib.request
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import librosa
import numpy as np
import torch
from datasets import load_dataset, Dataset, Audio as HFAudio
from dotenv import load_dotenv, find_dotenv
from groq import Groq
from IPython.display import Audio, display
from panns_inference import AudioTagging
from pinecone import Pinecone, ServerlessSpec
from tqdm.auto import tqdm

# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [ ]:
# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [ ]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [ ]:
# select only the audio data from the dataset and store in a numpy array
audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [2]:
panns_dir = Path.home() / "panns_data"
panns_dir.mkdir(exist_ok=True)

files = {
    "class_labels_indices.csv": "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv",
    "Cnn14_mAP=0.431.pth": "https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth?download=1"
}

for filename, url in files.items():
    path = panns_dir / filename
    if not path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, path)
        print("Done.")
    else:
        print(f"{filename} already exists, skipping.")

class_labels_indices.csv already exists, skipping.
Cnn14_mAP=0.431.pth already exists, skipping.


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = AudioTagging(checkpoint_path=None, device=device)

Using device: cuda
Checkpoint path: C:\Users\Work/panns_data/Cnn14_mAP=0.431.pth
GPU number: 1


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [4]:
_ = load_dotenv(find_dotenv())

# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

In [5]:
# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [6]:
cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [7]:
index_name = "audio-search-demo"

In [8]:
# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 02 Mar 2026 16:19:42 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '59',
                                    'x-pinecone-request-latency-ms': '58',
                                    'x-pinecone-response-duration-ms': '60'}},
 'dimension': 2048,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [33]:
# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/8 [00:00<?, ?it/s]

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '186',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 02 Mar 2026 16:32:08 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '6'}},
 'dimension': 2048,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 500}},
 'storageFullness': 0.0,
 'total_vector_count': 500,
 'vector_type': 'dense'}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [ ]:
# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

In [ ]:
# reshape query audio and generate embedding
query_audio = query_audio[None, :]
_, xq = model.inference(query_audio)
xq.shape

In [ ]:
# query pinecone index with the query audio embeddings
# top_k=4 to account for the self-match which is always the first result
results = index.query(vector=xq.tolist(), top_k=4)
results

In [ ]:
# play the top 3 similar audios (skip index 0 — always the query itself)
for r in results["matches"][1:]:
    a = data[int(r["id"])]["audio"]["array"]
    a = a / np.max(np.abs(a)) * 0.3
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [ ]:
def find_similar_audios(id):
    print("Query Audio:")
    query_audio = data[id]["audio"]["array"]
    display(Audio(query_audio, rate=44100))
    # top_k=6 to get 5 results after skipping the self-match
    result = index.query(id=str(id), top_k=6)
    print("Results:")
    for r in result["matches"][1:]:  # skip index 0 — always the query itself
        a = data[int(r["id"])]["audio"]["array"]
        a = a / np.max(np.abs(a)) * 0.3
        display(Audio(a, rate=44100))

In [ ]:
find_similar_audios(1642)

In [ ]:
find_similar_audios(452)

And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [ ]:
url = "https://storage.googleapis.com/audioset/miaow_16k.wav"
output_path = "./data/miaow_16k.wav"

os.makedirs("./data", exist_ok=True)
if not os.path.exists(output_path):
    print("Downloading miaow_16k.wav...")
    urllib.request.urlretrieve(url, output_path)
    print("Done.")
else:
    print("File already exists, skipping.")

We can load the audio into a Numpy array as follows:

In [ ]:
a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [ ]:
# reshape query audio and generate embedding
query_audio = a[None, :]
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
# miaow_16k.wav is not in the index so no self-match to skip
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    a = a / np.max(np.abs(a)) * 0.3
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Improvement 0: Swap Dataset for Actual Music

The default ESC-50 dataset contains environmental sounds. To build a genuine music recommender, swap it for a music dataset.

**Three options are provided below** — uncomment exactly one option block, then run the normaliser cell beneath it.

The normaliser rebuilds `data`, `audios`, and `categories` using the same structure the rest of the notebook expects, so **Improvements 1 and 2 continue to work without any further changes**.

> **After swapping the dataset**, scroll back up to *Generate Embeddings and Upsert* and re-run it to populate Pinecone with the new dataset's embeddings. You may want to delete the existing index first (see the *Delete the Index* cell at the bottom).

In [31]:
# ─── OPTION A ─── GTZAN  ·  1 000 clips  ·  10 genres  ·  30 s  ·  22 050 Hz ──────────────────
# The classic music-genre benchmark — small, fast, and well-known.
# Genres: blues, classical, country, disco, hip-hop, jazz, metal, pop, reggae, rock
# ────────────────────────────────────────────────────────────────────────────────────────────────
# raw_data  = load_dataset("marsyas/gtzan", split="train", trust_remote_code=True)
# AUDIO_KEY = "file"     # path string — librosa loads it in the normaliser
# LABEL_KEY = "genre"    # ClassLabel — decoded to string automatically by the normaliser
# NAME_KEY  = None       # no filename column; IDs will be auto-generated

# ─── OPTION B ─── lewtun/music_genres  ·  200 clips  ·  8 genres  ·  30 s ──────────────────────
# Curated FMA subset used in HuggingFace tutorials.
# Loading only 200 clips (split slice) to keep memory usage manageable on 32 GB machines.
# Increase the slice (e.g. "train[:500]") if you have RAM to spare.
# Genres: Electronic, Experimental, Folk, Hip-Hop, Instrumental, International, Pop, Rock
# ────────────────────────────────────────────────────────────────────────────────────────────────
raw_data  = load_dataset("lewtun/music_genres", split="train[:500]")
AUDIO_KEY = "audio"
LABEL_KEY = "genre"    # string field — use "genre", NOT "genre_id" (that is a raw integer)
NAME_KEY  = "song_id"

# ─── OPTION C ─── Local folder  ·  YOUR OWN FILES  ·  any format ────────────────────────────────
# Use any audio files you have downloaded (Zenodo, FMA, etc.).
# Organise them so each sub-folder name is the genre/category label:
#
#   music/
#     jazz/song1.mp3  song2.wav …
#     rock/song1.mp3  song2.wav …
#     classical/…
#
# ────────────────────────────────────────────────────────────────────────────────────────────────
# from pathlib import Path
# audio_dir = "./music"   # ← point this at your folder
# raw_data = [
#     {"path": str(p), "genre": p.parent.name, "filename": p.name}
#     for ext in ("*.wav", "*.mp3", "*.flac", "*.ogg")
#     for p in sorted(Path(audio_dir).rglob(ext))
# ]
# print(f"Found {len(raw_data)} files in {len(set(r['genre'] for r in raw_data))} genre folders")
# AUDIO_KEY = "path"     # path string — librosa loads it in the normaliser
# LABEL_KEY = "genre"    # folder name used as category
# NAME_KEY  = "filename"

In [32]:
# Run this cell after uncommenting exactly ONE option block above.
#
# What it does:
#   • Disables HuggingFace auto-decoding so full 30-s arrays are never loaded into RAM
#   • Loads each clip on demand via librosa (handles path strings, bytes, or decoded arrays)
#   • Resamples to 44 100 Hz, converts stereo → mono, trims / zero-pads to 5 s
#   • Normalises field names so all downstream code stays unchanged:
#       data[i]["audio"]["array"]  ·  data[i]["category"]  ·  data[i]["filename"]
#   • Rebuilds `audios` (numpy array) and `categories` (sorted list)
#
# After running this cell, scroll back to "Generate Embeddings and Upsert"
# and re-run it so Pinecone holds embeddings for the new dataset.
# You may want to delete the existing index first (see the cell at the bottom).

SR      = 44100      # target sample rate (Hz)
MAX_LEN = 5 * SR     # 5-second clips

# Free memory from any previously loaded dataset before building new arrays
try:
    del data, audios
except NameError:
    pass
gc.collect()

# Disable automatic audio decoding for HuggingFace datasets.
# Without this, the library decodes the full 30-s array into RAM on every iteration
# before our normaliser gets a chance to trim it — causes MemoryError on large datasets.
if isinstance(raw_data, Dataset):
    try:
        raw_data = raw_data.cast_column(AUDIO_KEY, HFAudio(decode=False))
    except Exception:
        pass  # AUDIO_KEY is not an Audio column (e.g. GTZAN "file" is already a path string)

normalized = []
for i, item in enumerate(tqdm(raw_data, desc="Normalising audio")):
    audio_field = item[AUDIO_KEY]

    # Three possible formats:
    # 1. Decoded Audio dict   → {"array": ..., "sampling_rate": ...}  (ESC-50 style)
    # 2. Undecoded Audio dict → {"path": "...", "bytes": b"..."}       (after cast_column)
    # 3. Plain path string    → load directly with librosa             (GTZAN, local folder)
    if isinstance(audio_field, dict) and "array" in audio_field:
        arr = np.array(audio_field["array"], dtype=np.float32)
        sr  = audio_field["sampling_rate"]
    elif isinstance(audio_field, dict):
        bdata = audio_field.get("bytes")
        path  = audio_field.get("path")
        if bdata:
            arr, sr = librosa.load(io.BytesIO(bdata), sr=None)
        else:
            arr, sr = librosa.load(path, sr=None)
        arr = arr.astype(np.float32)
    else:
        arr, sr = librosa.load(audio_field, sr=None)
        arr = arr.astype(np.float32)

    # Resample if the source rate differs from the target
    if sr != SR:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=SR)

    # Convert stereo to mono
    if arr.ndim > 1:
        arr = arr.mean(axis=0).astype(np.float32)

    # Trim to MAX_LEN or zero-pad if shorter
    if len(arr) > MAX_LEN:
        arr = arr[:MAX_LEN]
    elif len(arr) < MAX_LEN:
        arr = np.pad(arr, (0, MAX_LEN - len(arr)))

    # Decode ClassLabel integer → string if the field returns an int
    cat = item[LABEL_KEY]
    if not isinstance(cat, str):
        try:
            cat = raw_data.features[LABEL_KEY].int2str(cat)
        except Exception:
            cat = str(cat)

    fname = (
        str(item[NAME_KEY])
        if NAME_KEY and NAME_KEY in item
        else f"track_{i:05d}"
    )

    normalized.append({
        "audio":    {"array": arr, "sampling_rate": SR},
        "category": cat,
        "filename": fname,
    })

# Overwrite the dataset variables used throughout the notebook
data       = normalized
audios     = np.array([item["audio"]["array"] for item in data], dtype=np.float32)
categories = sorted(set(item["category"] for item in data))

print(f"Dataset ready : {len(data)} clips  ·  {len(categories)} categories")
print(f"Audio shape   : {audios[0].shape}  ({SR} Hz, {MAX_LEN // SR} s)")
print(f"Categories    : {categories}")

Normalising audio:   0%|          | 0/500 [00:00<?, ?it/s]

Dataset ready : 500 clips  ·  18 categories
Audio shape   : (220500,)  (44100 Hz, 5 s)
Categories    : ['Ambient Electronic', 'Blues', 'Chiptune / Glitch', 'Classical', 'Country', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Jazz', 'Old-Time / Historic', 'Pop', 'Punk', 'Rock', 'Soul-RnB', 'Spoken']


# Improvement 1: Rich Metadata Queries

So far our index stores only raw audio embeddings — there's no way to filter results by category, artist, or any other attribute. By attaching metadata to each vector at upsert time, Pinecone lets us apply filters at query time so results stay within a desired category.

We'll re-upsert all 2000 vectors with their ESC-50 metadata: `category`, `filename`, and `fold`.

In [34]:
# Re-upsert all vectors with metadata (category, filename, fold)
# This updates the existing vectors in-place — no need to recreate the index
for i in tqdm(range(0, len(audios), batch_size)):
    i_end = min(i + batch_size, len(audios))
    batch = audios[i:i_end]
    _, emb = model.inference(batch)

    vectors = [
        (
            str(idx),
            emb[j].tolist(),
            {
                "category": data[idx]["category"],
                "filename": data[idx]["filename"],
                "fold": int(data[idx].get("fold", 0))  # ESC-50 has fold; music datasets default to 0
            }
        )
        for j, idx in enumerate(range(i, i_end))
    ]
    index.upsert(vectors=vectors)

print("All vectors re-upserted with metadata.")
index.describe_index_stats()

  0%|          | 0/8 [00:00<?, ?it/s]

All vectors re-upserted with metadata.


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '186',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 02 Mar 2026 16:32:21 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '38',
                                    'x-pinecone-request-latency-ms': '37',
                                    'x-pinecone-response-duration-ms': '40'}},
 'dimension': 2048,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 800}},
 'storageFullness': 0.0,
 'total_vector_count': 800,
 'vector_type': 'dense'}

In [25]:
# Change filter_category to a category that exists in your current dataset.
# Run: print(categories)  to see all available values.
# ESC-50 (default): "cat"  |  GTZAN: "jazz"  |  lewtun/music_genres: check categories
filter_category = "Experimental"

# Validate the category exists in the loaded data
matching_ids = [i for i, item in enumerate(data) if item["category"] == filter_category]
if not matching_ids:
    print(f"Category '{filter_category}' not found in current dataset.")
    print(f"Available categories: {categories}")
else:
    # Pick a random clip from that category as the query vector
    seed_id = random.choice(matching_ids)
    query_audio = data[seed_id]["audio"]["array"][None, :]
    _, xq = model.inference(query_audio)
    print(f"Query seed: #{seed_id} — {filter_category}")

    results_filtered = index.query(
        vector=xq.tolist(),
        top_k=4,
        filter={"category": {"$eq": filter_category}},
        include_metadata=True
    )

    print(f"\nFiltered results (category: {filter_category}):")
    for r in results_filtered["matches"][1:]:  # skip self-match
        meta = r["metadata"]
        print(f"  ID: {r['id']} | Score: {r['score']:.4f} | {meta['category']} — {meta['filename']}")
        a = data[int(r["id"])]["audio"]["array"]
        a = a / np.max(np.abs(a)) * 0.3
        display(Audio(a, rate=44100))

Query seed: #133 — Experimental

Filtered results (category: Experimental):
  ID: 169 | Score: 0.7316 | Experimental — 229


  ID: 188 | Score: 0.7307 | Experimental — 249


  ID: 109 | Score: 0.7101 | Experimental — 146


# Improvement 2: Natural Language Recommendations

Instead of querying by example audio, we can let users describe what they want in plain text. An LLM maps the description to the closest ESC-50 category, picks a random seed audio from that category, and queries Pinecone (with a metadata filter) to return similar recommendations.

In [35]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

# Works for both HuggingFace Dataset (ESC-50) and plain list (music datasets via normaliser)
categories = sorted(set(item["category"] for item in data))
print("Available categories:", categories)

Available categories: ['Ambient Electronic', 'Blues', 'Chiptune / Glitch', 'Classical', 'Country', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Jazz', 'Old-Time / Historic', 'Pop', 'Punk', 'Rock', 'Soul-RnB', 'Spoken']


In [36]:
def recommend_by_description(user_query: str, top_k: int = 3):
    """
    Takes a natural language description, uses an LLM to map it to an ESC-50 category,
    then queries Pinecone for similar audio recommendations.
    """
    # Step 1: LLM maps the description to the closest category
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": (
                    f"You are a sound expert. Given a user's description, pick the single most "
                    f"relevant category from this list: {', '.join(categories)}. "
                    "Respond with ONLY the exact category name, nothing else."
                )
            },
            {"role": "user", "content": user_query}
        ]
    )
    matched_category = response.choices[0].message.content.strip()

    # Validate the returned category
    if matched_category not in categories:
        matched_category = next(
            (c for c in categories if c.lower() in matched_category.lower()), categories[0]
        )

    print(f"Query: '{user_query}'")
    print(f"Matched category: '{matched_category}'")

    # Step 2: Pick a random seed audio from that category
    category_ids = [i for i, item in enumerate(data) if item["category"] == matched_category]
    seed_id = random.choice(category_ids)

    print(f"\nSeed audio (#{seed_id} — {matched_category}):")
    display(Audio(data[seed_id]["audio"]["array"], rate=44100))

    # Step 3: Query Pinecone filtered to the matched category
    results = index.query(
        id=str(seed_id),
        top_k=top_k + 1,
        filter={"category": {"$eq": matched_category}},
        include_metadata=True
    )

    print(f"\nTop {top_k} recommendations:")
    for r in results["matches"][1:]:  # skip self-match
        meta = r.get("metadata", {})
        print(f"  {meta.get('category', '?')} — {meta.get('filename', '?')} (score: {r['score']:.3f})")
        a = data[int(r["id"])]["audio"]["array"]
        a = a / np.max(np.abs(a)) * 0.3
        display(Audio(a, rate=44100))

In [37]:
recommend_by_description("something energetic to work out to")

Query: 'something energetic to work out to'
Matched category: 'Hip-Hop'

Seed audio (#152 — Hip-Hop):



Top 3 recommendations:
  Hip-Hop — 567 (score: 0.891)


  Hip-Hop — 53 (score: 0.889)


  Hip-Hop — 16 (score: 0.886)


In [38]:
recommend_by_description("calm acoustic background music")

Query: 'calm acoustic background music'
Matched category: 'Ambient Electronic'

Seed audio (#208 — Ambient Electronic):



Top 3 recommendations:
  Ambient Electronic — 521 (score: 0.895)


  Ambient Electronic — 362 (score: 0.880)


  Ambient Electronic — 310 (score: 0.876)


In [41]:
recommend_by_description("something melodic and groovy")

Query: 'something melodic and groovy'
Matched category: 'Pop'

Seed audio (#319 — Pop):



Top 3 recommendations:
  Pop — 369 (score: 0.955)


  Pop — 248 (score: 0.867)


  Pop — 130 (score: 0.860)


# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [ ]:
#pc.delete_index(index_name)

# Conclusions

## What We Built
We built an **audio similarity search and recommendation system** using two core components:
- **PANNs (Pretrained Audio Neural Networks)** — the Cnn14 model converts raw audio waveforms into rich 2048-dimensional embeddings that capture the acoustic fingerprint of each sound
- **Pinecone** — a managed vector database that indexes those embeddings and retrieves the most acoustically similar ones in milliseconds via cosine similarity

The base system works like a lightweight Shazam: given any audio clip — whether from the dataset or an external file — it finds the most similar sounds in the catalog.

---

## Extensions We Implemented

### Improvement 0 — Swap Dataset for Actual Music
Added a dataset-agnostic normaliser that allows swapping ESC-50 for real music datasets with a single uncommented block. Three options were provided (GTZAN, lewtun/music_genres, local folder). The normaliser handles resampling to 44 100 Hz, stereo-to-mono conversion, and 5-second trimming, and rebuilds `data`, `audios`, and `categories` so all downstream code continues to work unchanged.

### Improvement 1 — Rich Metadata Queries
Re-upserted all vectors with metadata (`category`, `filename`, `fold`) attached to each Pinecone record. This enables **filtered similarity search** — results can be constrained to a specific genre or category at query time without scanning the full index.

### Improvement 2 — Natural Language Recommendations
Integrated a Groq-hosted LLM (`llama-3.1-8b-instant`) as a semantic layer on top of the vector search. Users describe what they want in plain text; the LLM maps the description to the closest category in the dataset; Pinecone returns acoustically similar clips within that category. This combines **semantic understanding** (LLM) with **acoustic similarity** (vector search).

---

## What We Learned

| Topic | Key Insight |
|-------|------------|
| **Audio embeddings** | A pretrained model turns a raw waveform into a fixed-size vector — the geometry of that space encodes acoustic similarity |
| **Vector databases** | Cosine similarity over high-dimensional embeddings is fast enough for real-time search across thousands of records |
| **Self-match** | Querying with a vector already in the index always returns itself as the top result — skip `matches[0]` when the query is part of the dataset |
| **Memory management** | HuggingFace decodes full audio arrays eagerly — `cast_column(Audio(decode=False))` + `del` + `gc.collect()` are essential when RAM is limited |
| **Dataset normalisation** | Different datasets use different field names, sample rates, and label types — a normaliser that abstracts these differences keeps downstream code clean |
| **LLM as a semantic router** | LLMs are effective at mapping free-form text to structured categories, acting as a bridge between a user's intent and a structured retrieval system |
| **Metadata filtering** | Attaching metadata at upsert time is cheap; filtering at query time avoids returning irrelevant results even when embeddings are acoustically close |